Notebook 3: Landsat Feature Extraction\
This notebook extracts satellite imagery features from Landsat for all 9,319 water quality sample locations in South Africa. For each location and date it pulls a 100-meter buffer around the sample coordinates, finds the nearest cloud-free (<10% cloud cover) Landsat scene to the sample date, extracts 6 spectral bands using the median pixel values within that 100 m zone, and then calculates 14 more features.\
Final output is a single file (landsat_features.csv) with 23 columns that will be used alongside the terra climate data for water quality prediction modeling

In [ ]:
!pip install uv
!uv pip install  -r requirements.txt 

Cell 1: Install packages

In [ ]:
import subprocess
import sys

# Install required packages using subprocess (Snowflake-compatible)
packages = [
    'pystac-client',
    'planetary-computer',
    'odc-stac',
    'tqdm'
]

for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', package, '-q'])

print("All dependencies installed successfully")

Cell 2: Install Libraries

In [ ]:
# Snowflake session
from snowflake.snowpark.context import get_active_session
session = get_active_session()

import warnings
warnings.filterwarnings("ignore")

# Data manipulation
import numpy as np
import pandas as pd
from datetime import datetime
import os

# Planetary Computer and STAC
import pystac_client
import planetary_computer as pc
from odc.stac import stac_load

# Progress tracking
from tqdm import tqdm
tqdm.pandas()

# Download utilities
import base64
from IPython.display import HTML, display

print("All libraries imported successfully")
print(f"Working directory: {os.getcwd()}")

Cell 3: Load input data

In [ ]:
# Load unique samples
input_df = pd.read_csv('unique_samples_for_extraction.csv')

print(f"Total samples: {len(input_df):,}")
print(f"Unique locations: {input_df[['Latitude', 'Longitude']].drop_duplicates().shape[0]:,}")
print(f"Date range: {input_df['Sample Date'].min()} to {input_df['Sample Date'].max()}")
print(f"\nFirst few rows:")
display(input_df.head())
print(f"\nColumns: {list(input_df.columns)}")
print(f"\nMissing values:\n{input_df.isnull().sum()}")

Cell 4: Extraction Function

In [ ]:
# Initialize catalog once (shared across all calls)
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=pc.sign_inplace,
)

def extract_landsat_bands(row):
    """
    Extract all 6 Landsat bands for a single sample location.
    
    Parameters:
    -----------
    row : pandas.Series
        Must contain 'Latitude', 'Longitude', and 'Sample Date'
    
    Returns:
    --------
    pandas.Series with bands: nir, green, red, blue, swir16, swir22
    """
    lat = row['Latitude']
    lon = row['Longitude']
    date = pd.to_datetime(row['Sample Date'], dayfirst=True, errors='coerce')
    
    # Create 100m buffer around point
    bbox_size = 0.00089831  # ~100m at equator
    bbox = [
        lon - bbox_size / 2,
        lat - bbox_size / 2,
        lon + bbox_size / 2,
        lat + bbox_size / 2
    ]
    
    # Search for Landsat scenes
    try:
        search = catalog.search(
            collections=["landsat-c2-l2"],
            bbox=bbox,
            datetime="2011-01-01/2015-12-31",
            query={"eo:cloud_cover": {"lt": 10}},  # <10% cloud cover
        )
        
        items = search.item_collection()
        
        if not items:
            # No scenes found
            return pd.Series({
                "nir": np.nan, "green": np.nan, "red": np.nan,
                "blue": np.nan, "swir16": np.nan, "swir22": np.nan
            })
        
        # Convert sample date to UTC
        sample_date_utc = date.tz_localize("UTC") if date.tzinfo is None else date.tz_convert("UTC")
        
        # Select scene closest to sample date
        items_sorted = sorted(
            items,
            key=lambda x: abs(pd.to_datetime(x.properties["datetime"]).tz_convert("UTC") - sample_date_utc)
        )
        selected_item = pc.sign(items_sorted[0])
        
        # Load all 6 bands
        bands_of_interest = ["nir08", "green", "red", "blue", "swir16", "swir22"]
        data = stac_load([selected_item], bands=bands_of_interest, bbox=bbox).isel(time=0)
        
        # Extract median values
        nir = data["nir08"].astype("float")
        green = data["green"].astype("float")
        red = data["red"].astype("float")
        blue = data["blue"].astype("float")
        swir16 = data["swir16"].astype("float")
        swir22 = data["swir22"].astype("float")
        
        # Compute medians (skipna=True handles internal NaNs)
        median_nir = float(nir.median(skipna=True).values)
        median_green = float(green.median(skipna=True).values)
        median_red = float(red.median(skipna=True).values)
        median_blue = float(blue.median(skipna=True).values)
        median_swir16 = float(swir16.median(skipna=True).values)
        median_swir22 = float(swir22.median(skipna=True).values)
        
        # Replace 0 with NaN (invalid Landsat values)
        median_nir = median_nir if median_nir != 0 else np.nan
        median_green = median_green if median_green != 0 else np.nan
        median_red = median_red if median_red != 0 else np.nan
        median_blue = median_blue if median_blue != 0 else np.nan
        median_swir16 = median_swir16 if median_swir16 != 0 else np.nan
        median_swir22 = median_swir22 if median_swir22 != 0 else np.nan
        
        return pd.Series({
            "nir": median_nir,
            "green": median_green,
            "red": median_red,
            "blue": median_blue,
            "swir16": median_swir16,
            "swir22": median_swir22,
        })
        
    except Exception as e:
        # Handle any errors gracefully
        return pd.Series({
            "nir": np.nan, "green": np.nan, "red": np.nan,
            "blue": np.nan, "swir16": np.nan, "swir22": np.nan
        })

print("Landsat extraction function defined")
print("Connected to Microsoft Planetary Computer")
print("Data source: Landsat Collection 2 Level 2")

Cell 5: Testing extraction b/ it takes a while make sure code is good

In [ ]:
# Test on first 5 rows
print("Testing extraction on first 5 samples\n")

test_df = input_df.head(5).copy()
test_results = test_df.progress_apply(extract_landsat_bands, axis=1)

print("\nTest extraction complete\n")
print("Test Results:")
display(test_results)

# Check for any data
nan_count = test_results.isna().sum().sum()
total_values = test_results.size
valid_values = total_values - nan_count

print(f"\nData Quality:")
print(f"   Valid values: {valid_values}/{total_values} ({100*valid_values/total_values:.1f}%)")
print(f"   Missing values: {nan_count}/{total_values} ({100*nan_count/total_values:.1f}%)")

if valid_values > 0:
    print("\nExtraction function working correctly! Ready to process full dataset.")
else:
    print("\nWARNING: No valid data extracted. Check API connection and coordinates.")

Cell 6: Full extraction (in batches)

In [ ]:
# Configuration
BATCH_SIZE = 200
STAGE_NAME = "LANDSAT_BATCHES"

total_samples = len(input_df)
total_batches = (total_samples + BATCH_SIZE - 1) // BATCH_SIZE

# Create stage if it doesn't exist
try:
    session.sql(f"CREATE STAGE IF NOT EXISTS {STAGE_NAME}").collect()
    print(f"✅ Stage {STAGE_NAME} ready for batch storage\n")
except:
    pass

print(f"🚀 Starting Landsat Feature Extraction")
print(f"━" * 60)
print(f"📊 Total samples: {total_samples:,}")
print(f"📦 Batch size: {BATCH_SIZE}")
print(f"🔢 Total batches: {total_batches}")
print(f"━" * 60)

# Check which batches already exist in stage
existing_batches = set()
try:
    files = session.sql(f"LIST @{STAGE_NAME}").collect()
    for file in files:
        filename = file['name']
        if 'landsat_batch_' in filename:
            parts = filename.split('_')
            if len(parts) >= 3:
                batch_start = parts[-2]
                existing_batches.add(batch_start)
    print(f"✅ Found {len(existing_batches)} existing batches in Snowflake storage\n")
except:
    print(f"📝 No existing batches found - starting fresh\n")

# Process each batch
for batch_num in range(total_batches):
    batch_start = batch_num * BATCH_SIZE
    batch_end = min(batch_start + BATCH_SIZE, total_samples)
    
    # Check if already completed
    if str(batch_start) in existing_batches:
        print(f"⏭️  Batch {batch_num+1}/{total_batches} - Already completed")
        continue
    
    print(f"\n🔄 Processing Batch {batch_num+1}/{total_batches} [{batch_start}-{batch_end}]")
    
    try:
        # Extract batch WITH SIMPLE PROGRESS
        batch_df = input_df.iloc[batch_start:batch_end].copy()
        
        results_list = []
        for idx, (_, row) in enumerate(batch_df.iterrows()):
            result = extract_landsat_bands(row)
            results_list.append(result)
            
            # Print every 10 samples
            if (idx + 1) % 10 == 0:
                print(f"   ... processed {idx + 1}/{len(batch_df)} samples")
        
        batch_results = pd.DataFrame(results_list)
        
        # Save to /tmp first
        temp_path = f"/tmp/landsat_batch_{batch_start}_{batch_end}.csv"
        batch_results.to_csv(temp_path, index=False)
        
        # Upload to Snowflake stage (PERSISTENT!)
        session.sql(f"""
            PUT file://{temp_path}
            @{STAGE_NAME}
            AUTO_COMPRESS=FALSE
            OVERWRITE=TRUE
        """).collect()
        
        # Report statistics
        nan_count = batch_results.isna().sum().sum()
        total_values = batch_results.size
        valid_pct = 100 * (1 - nan_count/total_values)
        
        print(f"✅ Batch {batch_num+1}/{total_batches} complete - {valid_pct:.1f}% valid data")
        print(f"💾 Saved to Snowflake stage: @{STAGE_NAME}")
        
    except Exception as e:
        print(f"❌ ERROR in Batch {batch_num+1}/{total_batches}: {str(e)}")
        print(f"⚠️  Re-run this cell to resume")
        break

print(f"\n{'━' * 60}")
print(f"🎉 Extraction phase complete!")
print(f"➡️  Proceed to Cell 15 to combine batches")

In [ ]:
# Check how many batches are saved
files = session.sql("LIST @LANDSAT_BATCHES").collect()
print(f"✅ {len(files)} batches currently saved:")
for f in files:
    name = f['name'].split('/')[-1]  # Get just the filename
    print(f"   {name}")

In [ ]:
# Configuration
BATCH_SIZE = 200
STAGE_NAME = "LANDSAT_BATCHES"

total_samples = len(input_df)
total_batches = (total_samples + BATCH_SIZE - 1) // BATCH_SIZE

# Create stage if it doesn't exist
try:
    session.sql(f"CREATE STAGE IF NOT EXISTS {STAGE_NAME}").collect()
except:
    pass

print(f"🚀 Starting Landsat Feature Extraction")
print(f"━" * 60)
print(f"📊 Total samples: {total_samples:,}")
print(f"📦 Batch size: {BATCH_SIZE}")
print(f"🔢 Total batches: {total_batches}")
print(f"━" * 60)

# Check which batches already exist
existing_batches = set()
try:
    files = session.sql(f"LIST @{STAGE_NAME}").collect()
    for file in files:
        filename = file['name']
        if 'landsat_batch_' in filename:
            parts = filename.split('_')
            if len(parts) >= 3:
                batch_start = parts[-2]
                existing_batches.add(batch_start)
    print(f"✅ Found {len(existing_batches)} existing batches\n")
except:
    print(f"📝 No existing batches found\n")

# Process each batch
for batch_num in range(total_batches):
    batch_start = batch_num * BATCH_SIZE
    batch_end = min(batch_start + BATCH_SIZE, total_samples)
    
    # Check if already completed
    if str(batch_start) in existing_batches:
        print(f"⏭️  Batch {batch_num+1}/{total_batches} - Already completed")
        continue
    
    print(f"\n🔄 Processing Batch {batch_num+1}/{total_batches} [{batch_start}-{batch_end}]")
    
    # Check if partial progress exists
    temp_path = f"/tmp/landsat_batch_{batch_start}_{batch_end}.csv"
    
    try:
        session.sql(f"GET @{STAGE_NAME}/landsat_batch_{batch_start}_{batch_end}.csv file:///tmp/").collect()
        if os.path.exists(temp_path):
            partial_results = pd.read_csv(temp_path)
            start_idx = len(partial_results)
            print(f"   📂 Resuming from sample {start_idx}/{batch_end - batch_start}")
        else:
            partial_results = pd.DataFrame()
            start_idx = 0
    except:
        partial_results = pd.DataFrame()
        start_idx = 0
    
    try:
        batch_df = input_df.iloc[batch_start:batch_end].copy()
        results_list = [] if start_idx == 0 else partial_results.to_dict('records')
        
        for idx in range(start_idx, len(batch_df)):
            row = batch_df.iloc[idx]
            result = extract_landsat_bands(row)
            results_list.append(result.to_dict())
            
            # Save every 10 samples (silently)
            if (idx + 1) % 10 == 0:
                batch_results = pd.DataFrame(results_list)
                batch_results.to_csv(temp_path, index=False)
                
                session.sql(f"""
                    PUT file://{temp_path}
                    @{STAGE_NAME}
                    AUTO_COMPRESS=FALSE
                    OVERWRITE=TRUE
                """).collect()
                
                # Only PRINT every 50 samples (not every 10)
                if (idx + 1) % 50 == 0:
                    print(f"   💾 Progress: {idx + 1}/{len(batch_df)} samples saved")
        
        # Final save
        batch_results = pd.DataFrame(results_list)
        batch_results.to_csv(temp_path, index=False)
        
        session.sql(f"""
            PUT file://{temp_path}
            @{STAGE_NAME}
            AUTO_COMPRESS=FALSE
            OVERWRITE=TRUE
        """).collect()
        
        nan_count = batch_results.isna().sum().sum()
        total_values = batch_results.size
        valid_pct = 100 * (1 - nan_count/total_values)
        
        print(f"✅ Batch {batch_num+1}/{total_batches} complete - {valid_pct:.1f}% valid data")
        
    except Exception as e:
        print(f"❌ ERROR in Batch {batch_num+1}/{total_batches}: {str(e)}")
        print(f"⚠️  Re-run this cell to resume")
        break

print(f"\n{'━' * 60}")
print(f"🎉 Extraction phase complete!")

Cell 7: Combine batches and also calculating new features,
new features are:
Chlorophyll_Proxy
BSI
SAVI
AWEI_nsh
AWEI_sh
NRI
Turbidity_Index
NDVI
MNDWI
NDMI
NDTI
EVI
NDBI
NBR

In [ ]:
# Download all batches from Snowflake stage
STAGE_NAME = "LANDSAT_BATCHES"
BATCH_SIZE = 200
total_samples = len(input_df)
total_batches = (total_samples + BATCH_SIZE - 1) // BATCH_SIZE

print("🔄 Downloading batches from Snowflake storage...")

# Get all batches to /tmp
session.sql(f"""
    GET @{STAGE_NAME} file:///tmp/
""").collect()

print("✅ Downloaded all batches\n")

# Check for missing batches
missing_batches = []
batch_dfs = []

for batch_num in range(total_batches):
    batch_start = batch_num * BATCH_SIZE
    batch_end = min(batch_start + BATCH_SIZE, total_samples)
    batch_path = f"/tmp/landsat_batch_{batch_start}_{batch_end}.csv"
    
    if not os.path.exists(batch_path):
        missing_batches.append(f"{batch_start}-{batch_end}")
    else:
        batch_dfs.append(pd.read_csv(batch_path))

if missing_batches:
    print(f"⚠️  WARNING: Missing batches: {missing_batches}")
    print(f"Please re-run Cell 13 to complete extraction")
else:
    print(f"✅ All {total_batches} batches found!\n")
    
    # Combine all batches
    landsat_bands = pd.concat(batch_dfs, ignore_index=True)
    
    print(f"✅ Combined {total_batches} batches")
    print(f"📊 Total rows: {len(landsat_bands):,}")
    
    # Add location and date columns
    landsat_bands['Latitude'] = input_df['Latitude'].values
    landsat_bands['Longitude'] = input_df['Longitude'].values
    landsat_bands['Sample Date'] = input_df['Sample Date'].values
    
    print(f"\n🧮 Calculating water quality indices...")
    
    # Small epsilon to avoid division by zero
    eps = 1e-10
    
    # Water Quality Indices (from colleague's work)
    landsat_bands['Chlorophyll_Proxy'] = (
        (landsat_bands['nir'] - landsat_bands['red']) / 
        (landsat_bands['nir'] + landsat_bands['red'] + eps)
    )
    
    landsat_bands['BSI'] = (
        ((landsat_bands['swir16'] + landsat_bands['red']) - (landsat_bands['nir'] + landsat_bands['blue'])) / 
        ((landsat_bands['swir16'] + landsat_bands['red']) + (landsat_bands['nir'] + landsat_bands['blue']) + eps)
    )
    
    landsat_bands['SAVI'] = (
        ((landsat_bands['nir'] - landsat_bands['red']) / 
         (landsat_bands['nir'] + landsat_bands['red'] + 0.5 + eps)) * 1.5
    )
    
    landsat_bands['AWEI_nsh'] = (
        4 * (landsat_bands['green'] - landsat_bands['swir16']) - 
        (0.25 * landsat_bands['nir'] + 2.75 * landsat_bands['swir22'])
    )
    
    landsat_bands['AWEI_sh'] = (
        landsat_bands['blue'] + 2.5 * landsat_bands['green'] - 
        1.5 * (landsat_bands['nir'] + landsat_bands['swir16']) - 
        0.25 * landsat_bands['swir22']
    )
    
    landsat_bands['NRI'] = (
        (landsat_bands['green'] - landsat_bands['red']) / 
        (landsat_bands['green'] + landsat_bands['red'] + eps)
    )
    
    landsat_bands['Turbidity_Index'] = (
        landsat_bands['red'] / (landsat_bands['blue'] + eps)
    )
    
    # Standard Vegetation/Water Indices
    landsat_bands['NDVI'] = (
        (landsat_bands['nir'] - landsat_bands['red']) / 
        (landsat_bands['nir'] + landsat_bands['red'] + eps)
    )
    
    landsat_bands['MNDWI'] = (
        (landsat_bands['green'] - landsat_bands['swir16']) / 
        (landsat_bands['green'] + landsat_bands['swir16'] + eps)
    )
    
    landsat_bands['NDMI'] = (
        (landsat_bands['nir'] - landsat_bands['swir16']) / 
        (landsat_bands['nir'] + landsat_bands['swir16'] + eps)
    )
    
    landsat_bands['NDTI'] = (
        (landsat_bands['red'] - landsat_bands['green']) / 
        (landsat_bands['red'] + landsat_bands['green'] + eps)
    )
    
    landsat_bands['EVI'] = (
        2.5 * (landsat_bands['nir'] - landsat_bands['red']) / 
        (landsat_bands['nir'] + 6*landsat_bands['red'] - 7.5*landsat_bands['blue'] + 1 + eps)
    )
    
    landsat_bands['NDBI'] = (
        (landsat_bands['swir16'] - landsat_bands['nir']) / 
        (landsat_bands['swir16'] + landsat_bands['nir'] + eps)
    )
    
    landsat_bands['NBR'] = (
        (landsat_bands['nir'] - landsat_bands['swir22']) / 
        (landsat_bands['nir'] + landsat_bands['swir22'] + eps)
    )
    
    print(f"✅ Calculated 14 indices")
    
    # Reorder columns
    column_order = [
        'Latitude', 'Longitude', 'Sample Date',
        'nir', 'green', 'red', 'blue', 'swir16', 'swir22',
        'Chlorophyll_Proxy', 'BSI', 'SAVI', 'AWEI_nsh', 'AWEI_sh', 'NRI', 'Turbidity_Index',
        'NDVI', 'MNDWI', 'NDMI', 'NDTI', 'EVI', 'NDBI', 'NBR'
    ]
    landsat_bands = landsat_bands[column_order]
    
    # Save final output
    output_path = "/tmp/landsat_features.csv"
    landsat_bands.to_csv(output_path, index=False)
    
    print(f"\n💾 Saved final file to: {output_path}")
    display(landsat_bands.head(10))
    print(f"\n✅ All processing complete! Proceed to Cell 17 for quality checks.")

Cell 7: Data Quality Checks

In [ ]:
# Load final dataset
landsat_df = pd.read_csv("/tmp/landsat_features.csv")

print("LANDSAT FEATURE EXTRACTION - DATA QUALITY REPORT")
print("=" * 70)

# 1. Basic Statistics
print("\n1️BASIC STATISTICS")
print("-" * 70)
print(f"Total samples: {len(landsat_df):,}")
print(f"Total features: {len(landsat_df.columns)}")
print(f"Date range: {landsat_df['Sample Date'].min()} to {landsat_df['Sample Date'].max()}")
print(f"Unique locations: {landsat_df[['Latitude', 'Longitude']].drop_duplicates().shape[0]:,}")

# 2. Missing Data Analysis
print("\n2️MISSING DATA ANALYSIS")
print("-" * 70)
missing_counts = landsat_df.isnull().sum()
missing_pcts = (missing_counts / len(landsat_df)) * 100
missing_summary = pd.DataFrame({
    'Missing Count': missing_counts,
    'Missing %': missing_pcts
}).sort_values('Missing %', ascending=False)

print("\nMissing values by feature:")
display(missing_summary[missing_summary['Missing Count'] > 0])

total_values = landsat_df.size
total_missing = landsat_df.isnull().sum().sum()
print(f"\nOverall completeness: {100*(1-total_missing/total_values):.2f}%")
print(f"Total missing values: {total_missing:,} out of {total_values:,}")

# 3. Band Value Ranges
print("\n3️RAW BAND VALUE RANGES")
print("-" * 70)
bands = ['nir', 'green', 'red', 'blue', 'swir16', 'swir22']
band_stats = landsat_df[bands].describe().T[['min', 'mean', 'max', '50%']]
band_stats.columns = ['Min', 'Mean', 'Max', 'Median']
display(band_stats)

# 4. Index Value Ranges
print("\n4️WATER QUALITY INDEX RANGES")
print("-" * 70)
wq_indices = ['Chlorophyll_Proxy', 'BSI', 'SAVI', 'AWEI_nsh', 'AWEI_sh', 'NRI', 'Turbidity_Index']
wq_stats = landsat_df[wq_indices].describe().T[['min', 'mean', 'max', '50%']]
wq_stats.columns = ['Min', 'Mean', 'Max', 'Median']
display(wq_stats)

print("\n5️STANDARD INDEX RANGES")
print("-" * 70)
std_indices = ['NDVI', 'MNDWI', 'NDMI', 'NDTI', 'EVI', 'NDBI', 'NBR']
std_stats = landsat_df[std_indices].describe().T[['min', 'mean', 'max', '50%']]
std_stats.columns = ['Min', 'Mean', 'Max', 'Median']
display(std_stats)

# 5. Samples with Complete Data
print("\n6️SAMPLE COMPLETENESS")
print("-" * 70)
feature_cols = [col for col in landsat_df.columns if col not in ['Latitude', 'Longitude', 'Sample Date']]
complete_samples = landsat_df[feature_cols].dropna()
print(f"Samples with all features: {len(complete_samples):,} ({100*len(complete_samples)/len(landsat_df):.1f}%)")
print(f"Samples with any missing: {len(landsat_df) - len(complete_samples):,} ({100*(len(landsat_df)-len(complete_samples))/len(landsat_df):.1f}%)")

# 6. Data Quality Flags
print("\n7️DATA QUALITY FLAGS")
print("-" * 70)

flags = []

# Check for unusual values
for band in bands:
    if landsat_df[band].max() > 50000:
        flags.append(f"{band}: Max value ({landsat_df[band].max():.0f}) unusually high")
    if landsat_df[band].min() < 0:
        flags.append(f"{band}: Negative values detected (min: {landsat_df[band].min():.0f})")

# Check for infinite values in indices
all_indices = wq_indices + std_indices
for idx in all_indices:
    if np.isinf(landsat_df[idx]).any():
        inf_count = np.isinf(landsat_df[idx]).sum()
        flags.append(f"{idx}: {inf_count} infinite values detected")

if flags:
    for flag in flags:
        print(flag)
else:
    print("No data quality issues detected!")

# 7. Geographic Distribution
print("\n8️GEOGRAPHIC DISTRIBUTION")
print("-" * 70)
print(f"Latitude range: {landsat_df['Latitude'].min():.4f}° to {landsat_df['Latitude'].max():.4f}°")
print(f"Longitude range: {landsat_df['Longitude'].min():.4f}° to {landsat_df['Longitude'].max():.4f}°")

print("\n" + "=" * 70)
print("Quality check complete")
print("=" * 70)

Cell 8: Download link

In [ ]:
# Load final dataset
landsat_df = pd.read_csv("/tmp/landsat_features.csv")

# Convert to CSV string
csv_string = landsat_df.to_csv(index=False)

# Encode to base64
b64_encoded = base64.b64encode(csv_string.encode()).decode()

# Create download link
download_link = f'<a href="data:file/csv;base64,{b64_encoded}" download="landsat_features.csv">📥 Download landsat_features.csv</a>'

print(" Landsat Feature Extraction Complete!")
print(f" Final dataset: {len(landsat_df):,} samples × {len(landsat_df.columns)} features")
print(f" File size: {len(csv_string):,} bytes ({len(csv_string)/1024/1024:.2f} MB)")


display(HTML(download_link))
